# Pittsburgh Food Safety: Data Pipeline
**95-815 Data Engineering · Group 7**

This notebook is the whole pipeline, from downloading the sources to producing cleaned staging tables. Run it top to bottom. Every step is safe to re-run.

| Step | Section | Where the data ends up |
|---|---|---|
| Acquire | 2. Download sources | `data/` folder (local, git-ignored) |
| Store | 3. Database layout | PostgreSQL schemas `raw`, `staging`, `serving`, `meta` |
| Ingest (tabular) | 4. CSV → PostgreSQL | `raw.*` tables (all `TEXT`, no constraints) |
| Ingest (semi-structured) | 5. GeoJSON → MongoDB | `group7.neighborhoods_geojson` |
| Flatten | 6. MongoDB → PostGIS | `staging.neighborhoods` |
| Quality | 7. Data-quality checks | `meta.dq_log` |
| Clean | 8. Initial cleaning | `staging.*` tables (typed, keyed, constrained) |
| Verify | 9. Results | before/after counts, constraints, samples |

**Before running:** copy `.env.example` to `.env` and fill in the credentials, then `pip install -r requirements.txt`.

## 1. Setup
Imports, configuration and database connections. Credentials are read from the git-ignored `.env` file and are never written in the notebook.

In [1]:
import csv, hashlib, json, os, re, sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import psycopg2
import requests
from dotenv import load_dotenv
from psycopg2 import sql as pgsql
from psycopg2.extras import Json, execute_values
from pymongo import MongoClient, ReplaceOne
from pymongo.errors import OperationFailure

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)
load_dotenv(ROOT / ".env")
csv.field_size_limit(sys.maxsize)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 60)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")  # tags this run's log entries
print("run id:", RUN_ID)

run id: 20261122T025344Z


In [2]:
# PostgreSQL (AWS RDS)
pg = psycopg2.connect(
    host=os.environ["PGHOST"], port=os.getenv("PGPORT", "5432"),
    dbname=os.getenv("PGDATABASE", "postgres"),
    user=os.environ["PGUSER"], password=os.environ["PGPASSWORD"],
    sslmode=os.getenv("PGSSLMODE", "require"), connect_timeout=15,
    application_name="pgh-food-safety-notebook",
)

def sql(query, params=None):
    "Run SQL in one transaction (commit on success, rollback on error). SELECT -> DataFrame, else row count."
    with pg:
        with pg.cursor() as cur:
            cur.execute(query, params)
            if cur.description:
                return pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])
            return cur.rowcount

# MongoDB (Atlas)
mongo = MongoClient(os.environ["MONGO_URI"], serverSelectionTimeoutMS=15000)
mdb = mongo[os.getenv("MONGO_DB", "group7")]
mdb.command("ping")

display(sql("SELECT current_user AS pg_user, current_database() AS pg_database, split_part(version(), ' on ', 1) AS pg_version"))
print("MongoDB connected, database:", mdb.name)

,pg_user,pg_database,pg_version
0,grp7_warriors,postgres,PostgreSQL 18.3


MongoDB connected, database: group7


## 2. Download sources
All four sources come from the Western Pennsylvania Regional Data Center (WPRDC). Three are CSV files published by the Allegheny County Health Department. The fourth is a GeoJSON file of neighborhood boundaries published by the City of Pittsburgh.

| Dataset | Grain | Joins to |
|---|---|---|
| Inspections 2014–2025 | one row per inspection | violations on `encounter`, facilities on `id` |
| Violations 2014–2025 | one row per violation cited | inspections on `encounter` |
| Geocoded facilities | one row per facility, with x/y | inspections and violations on `id`; neighborhoods by location |
| Neighborhood boundaries (GeoJSON) | one feature per neighborhood | facilities by point-in-polygon |

**Evaluated and excluded: Inspections 2025–present** (resource `1c9cf8d8`). It covers Aug 2025 onward, after the violations data ends (31 Jul 2025), and it uses the county's new ID system, so none of its inspections can be linked to a violation or to a facility ID. The project's questions depend on violations, so this file isn't loaded.

Files already in `data/` are skipped. Set `FORCE_DOWNLOAD = True` to fetch fresh copies.

In [3]:
WPRDC_DUMP = "https://data.wprdc.org/datastore/dump/{}"
CSV_SOURCES = {   # raw table name -> WPRDC resource id
    "inspections_2014_2025":    "4ea730d8-2bf9-4783-b0a5-b41ed687097e",
    "violations_2014_2025":     "1a1329e2-418c-4bd3-af2c-cc334e7559af",
    "geocoded_facilities":      "112a3821-334d-4f3f-ab40-4de1220b1a0a",
}
GEOJSON_URL = ("https://data.wprdc.org/dataset/e672f13d-71c4-4a66-8f38-710e75ed80a4/resource/"
               "4af8e160-57e9-4ebf-a501-76ca1b42fc99/download/neighborhoods.geojson")
GEOJSON_FILE = DATA_DIR / "neighborhoods.geojson"
FORCE_DOWNLOAD = False

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def download(url, dest):
    if dest.exists() and not FORCE_DOWNLOAD:
        return "already downloaded"
    tmp = dest.with_suffix(dest.suffix + ".part")
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
    if tmp.stat().st_size == 0:
        raise RuntimeError(f"empty download: {url}")
    tmp.replace(dest)
    return "downloaded"

targets = [(name, WPRDC_DUMP.format(rid), DATA_DIR / f"{name}.csv") for name, rid in CSV_SOURCES.items()]
targets.append(("neighborhoods", GEOJSON_URL, GEOJSON_FILE))

rows = []
for name, url, dest in targets:
    status = download(url, dest)
    rows.append({"dataset": name, "file": dest.name, "size_mb": round(dest.stat().st_size / 1e6, 1),
                 "status": status, "source_url": url})
pd.DataFrame(rows)

,dataset,file,size_mb,status,source_url
0,inspections_2014_2025,inspections_2014_2025.csv,28.7,already downloaded,https://data.wprdc.org/datastore/dump/4ea730d8-2bf9-4783...
1,violations_2014_2025,violations_2014_2025.csv,121.6,already downloaded,https://data.wprdc.org/datastore/dump/1a1329e2-418c-4bd3...
2,geocoded_facilities,geocoded_facilities.csv,6.6,already downloaded,https://data.wprdc.org/datastore/dump/112a3821-334d-4f3f...
3,neighborhoods,neighborhoods.geojson,1.1,already downloaded,https://data.wprdc.org/dataset/e672f13d-71c4-4a66-8f38-7...


## 3. Database layout
All tabular data lives on **one PostgreSQL instance**, so every table can be joined in SQL. The layers are kept apart with schemas:

- **`raw`**: source data exactly as downloaded. Never edited by hand.
- **`staging`**: typed, cleaned, de-duplicated tables with keys and constraints.
- **`serving`**: analysis-ready tables (Milestone 3).
- **`meta`**: pipeline bookkeeping: a load log, a data-quality log and small helper functions.

PostGIS is enabled for the facility-to-neighborhood point-in-polygon join.

In [4]:
sql(r'''
CREATE SCHEMA IF NOT EXISTS raw;
CREATE SCHEMA IF NOT EXISTS staging;
CREATE SCHEMA IF NOT EXISTS serving;
CREATE SCHEMA IF NOT EXISTS meta;
CREATE EXTENSION IF NOT EXISTS postgis;

-- Tables from the excluded 2025-present file (loaded by an earlier version of this pipeline)
DROP TABLE IF EXISTS staging.inspections_2025, raw.inspections_2025_present;

CREATE TABLE IF NOT EXISTS meta.load_log (
    load_id       BIGSERIAL PRIMARY KEY,
    target_table  TEXT        NOT NULL,
    source_url    TEXT        NOT NULL,
    file_name     TEXT        NOT NULL,
    file_bytes    BIGINT      NOT NULL,
    file_sha256   TEXT        NOT NULL,
    csv_records   BIGINT      NOT NULL,
    rows_loaded   BIGINT      NOT NULL,
    loaded_by     TEXT        NOT NULL DEFAULT current_user,
    loaded_at     TIMESTAMPTZ NOT NULL DEFAULT now()
);
ALTER TABLE meta.load_log ADD COLUMN IF NOT EXISTS run_id TEXT;

CREATE TABLE IF NOT EXISTS meta.dq_log (
    check_id      BIGSERIAL PRIMARY KEY,
    table_name    TEXT        NOT NULL,
    check_name    TEXT        NOT NULL,
    severity      TEXT        NOT NULL CHECK (severity IN ('critical', 'warning', 'info')),
    affected_rows BIGINT      NOT NULL,
    action_taken  TEXT        NOT NULL,
    details       TEXT,
    run_by        TEXT        NOT NULL DEFAULT current_user,
    run_at        TIMESTAMPTZ NOT NULL DEFAULT now()
);
ALTER TABLE meta.dq_log ADD COLUMN IF NOT EXISTS run_id TEXT;

-- Safe casts: return NULL instead of failing on a malformed value, so bad values
-- can be counted by the quality checks and nulled out in staging.
CREATE OR REPLACE FUNCTION meta.try_date(t TEXT) RETURNS DATE LANGUAGE plpgsql IMMUTABLE AS $$
BEGIN RETURN NULLIF(btrim(t), '')::date; EXCEPTION WHEN others THEN RETURN NULL; END $$;
CREATE OR REPLACE FUNCTION meta.try_time(t TEXT) RETURNS TIME LANGUAGE plpgsql IMMUTABLE AS $$
BEGIN RETURN NULLIF(btrim(t), '')::time; EXCEPTION WHEN others THEN RETURN NULL; END $$;
CREATE OR REPLACE FUNCTION meta.try_int(t TEXT) RETURNS BIGINT LANGUAGE plpgsql IMMUTABLE AS $$
BEGIN RETURN NULLIF(btrim(t), '')::bigint; EXCEPTION WHEN others THEN RETURN NULL; END $$;
CREATE OR REPLACE FUNCTION meta.try_float(t TEXT) RETURNS DOUBLE PRECISION LANGUAGE plpgsql IMMUTABLE AS $$
BEGIN RETURN NULLIF(btrim(t), '')::double precision; EXCEPTION WHEN others THEN RETURN NULL; END $$;
''')
sql("SELECT schema_name FROM information_schema.schemata WHERE schema_name IN ('raw','staging','serving','meta') ORDER BY 1")

,schema_name
0,meta
1,raw
2,serving
3,staging


## 4. Ingest CSV files into PostgreSQL (`raw` schema)
The raw layer keeps the source data as close to the original as possible (requirement 10):
- **Every column is `TEXT`**, so a malformed date or number can't cause a row to be rejected.
- **No primary keys, `NOT NULL`, `UNIQUE` or `CHECK` constraints.** Those are added in `staging`.
- **Column names match the source headers.**
- **Blank fields load as `NULL`.**

Each table is fully reloaded with a bulk `COPY`. After loading, the table's row count is compared with the record count parsed from the file. **A mismatch is a critical error, and that table's load is rolled back.**

*Source quirk:* WPRDC files end the header line with CRLF but every data row with LF. `COPY` infers the line ending from the first line and rejects the mix, so the loader reads the header itself and streams the data rows unchanged.

In [5]:
def clean_col(name):
    return re.sub(r"[^0-9a-z_]+", "_", name.strip().lower()) or "col"   # keeps `_id` distinct from `id`

def load_csv_raw(table, resource_id):
    path, url = DATA_DIR / f"{table}.csv", WPRDC_DUMP.format(resource_id)
    with open(path, newline="", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        cols = [clean_col(h) for h in next(reader)]
        n_records = sum(1 for _ in reader)
    if len(set(cols)) != len(cols):
        raise RuntimeError(f"{path.name}: duplicate column names {cols}")

    ident = pgsql.Identifier("raw", table)
    with pg, pg.cursor() as cur:                       # one transaction per table
        cur.execute(pgsql.SQL("DROP TABLE IF EXISTS {}").format(ident))
        cur.execute(pgsql.SQL("CREATE TABLE {} ({})").format(
            ident, pgsql.SQL(", ").join(pgsql.SQL("{} TEXT").format(pgsql.Identifier(c)) for c in cols)))
        with open(path, "rb") as f:
            f.readline()                               # skip header (CRLF quirk, see above)
            cur.copy_expert(pgsql.SQL("COPY {} FROM STDIN WITH (FORMAT csv, ENCODING 'UTF8')")
                            .format(ident).as_string(pg), f)
        cur.execute(pgsql.SQL("SELECT count(*) FROM {}").format(ident))
        loaded = cur.fetchone()[0]
        if loaded != n_records:
            raise RuntimeError(f"CRITICAL: raw.{table} loaded {loaded:,} rows, file has {n_records:,}. Rolled back.")
        cur.execute(pgsql.SQL("COMMENT ON TABLE {} IS %s").format(ident), (f"Raw load of {url}",))
        cur.execute('''INSERT INTO meta.load_log (run_id, target_table, source_url, file_name, file_bytes,
                                                  file_sha256, csv_records, rows_loaded)
                       VALUES (%s, %s, %s, %s, %s, %s, %s, %s)''',
                    (RUN_ID, f"raw.{table}", url, path.name, path.stat().st_size, sha256(path), n_records, loaded))
    print(f"raw.{table:<26} {loaded:>9,} rows  ({len(cols)} columns)  OK")

for table, rid in CSV_SOURCES.items():
    load_csv_raw(table, rid)

raw.inspections_2014_2025        124,674 rows  (24 columns)  OK
raw.violations_2014_2025         405,474 rows  (22 columns)  OK
raw.geocoded_facilities           32,245 rows  (23 columns)  OK


**Load log for this run.** The record count parsed from each file matches the rows in the table.

In [6]:
sql('''SELECT target_table, file_name, round(file_bytes / 1e6, 1) AS file_mb, csv_records, rows_loaded,
              csv_records = rows_loaded AS counts_match, loaded_at
       FROM meta.load_log WHERE run_id = %s ORDER BY load_id''', (RUN_ID,))

,target_table,file_name,file_mb,csv_records,rows_loaded,counts_match,loaded_at
0,raw.inspections_2014_2025,inspections_2014_2025.csv,28.7,124674,124674,True,2026-11-22 02:53:46.414457+00:00
1,raw.violations_2014_2025,violations_2014_2025.csv,121.6,405474,405474,True,2026-11-22 02:53:51.342525+00:00
2,raw.geocoded_facilities,geocoded_facilities.csv,6.6,32245,32245,True,2026-11-22 02:54:22.511531+00:00


**Raw layer check.** Every raw column has type `TEXT` and no table has constraints.

In [7]:
sql('''
SELECT c.table_name, count(*) AS columns, string_agg(DISTINCT c.data_type, ', ') AS column_types,
       (SELECT count(*) FROM information_schema.table_constraints tc
         WHERE tc.table_schema = 'raw' AND tc.table_name = c.table_name
           AND tc.constraint_type IN ('PRIMARY KEY','UNIQUE','FOREIGN KEY','CHECK')) AS constraints
FROM information_schema.columns c
WHERE c.table_schema = 'raw' AND c.table_name IN ('inspections_2014_2025','violations_2014_2025','geocoded_facilities')
GROUP BY c.table_name ORDER BY 1
''')

,table_name,columns,column_types,constraints
0,geocoded_facilities,23,text,0
1,inspections_2014_2025,24,text,0
2,violations_2014_2025,22,text,0


**Sample rows from each raw table**

In [8]:
for t in CSV_SOURCES:
    print(f"raw.{t}")
    display(sql(f"SELECT * FROM raw.{t} LIMIT 3"))

raw.inspections_2014_2025


,_id,encounter,id,placard_st,placard_desc,facility_name,bus_st_date,category_cd,description,num,street,city,state,zip,inspect_dt,start_time,end_time,municipal,ispt_purpose,abrv,purpose,reispt_cd,reispt_dt,status
0,63603508,201704060020,10,1,Inspected & Permitted,Joe Winzek Catering Service,1970-01-01,100,Caterer,600,8th Street,Glassport,PA,15045,2017-04-06,13:00:00,13:25:00,Glassport,2,Reispt,Reinspection,2,NaN,1
1,63603509,201703010014,10,1,Inspected & Permitted,Joe Winzek Catering Service,1970-01-01,100,Caterer,600,8th Street,Glassport,PA,15045,2017-03-01,11:15:00,12:25:00,Glassport,1,Ini,Initial,1,2017-04-03,1
2,63603510,201410290023,10,1,Inspected & Permitted,Joe Winzek Catering Service,1970-01-01,100,Caterer,600,8th Street,Glassport,PA,15045,2014-10-29,13:15:00,14:15:00,Glassport,1,Ini,Initial,2,NaN,1


raw.violations_2014_2025


,_id,encounter,id,placard_st,facility_name,bus_st_date,description,description_new,num,street,city,state,zip,inspect_dt,start_time,end_time,municipal,rating,low,medium,high,url
0,287854015,201401020037,201202030011,1,Station Street Food,2012-02-01,Restaurant without Liquor,Certified Food Protection Manager,6290,Broad Street,Pittsburgh,PA,15206,2014-01-02,13:15:00,13:50:00,Pittsburgh-111,V,F,F,F,http://appsrv.alleghenycounty.us/reports/rwservlet?food_...
1,287854016,201401030015,201202090007,1,PaPa John's Pizza,2012-02-06,Restaurant without Liquor,Handwashing Facilities,1746,Wm Flynn Highway,Glenshaw,PA,15116,2014-01-03,13:45:00,14:30:00,Shaler,V,F,F,F,http://appsrv.alleghenycounty.us/reports/rwservlet?food_...
2,287854017,201401030016,200707240007,1,Giovanni's Pizza & Pasta,2007-07-16,Restaurant with Liquor,Hot Holding Temperatures,3047,West Liberty Avenue,Pittsburgh,PA,15216,2014-01-03,14:10:00,16:10:00,Dormont,N,F,F,F,http://appsrv.alleghenycounty.us/reports/rwservlet?food_...


raw.geocoded_facilities


,_id,id,facility_name,num,street,city,state,zip,municipal,category_cd,description,p_code,fdo,bus_st_date,bus_cl_date,seat_count,noroom,sq_feet,status,placard_st,x,y,address
0,1,57137,The Pizza Co,1409,Potomac Ave,Pittsburgh,PA,15216,Dormont,212,Chain Restaurant without Liquor,4,1998-02-03,1998-02-03,NaN,0,None,NaN,7,NaN,-80.034597,40.395249,1409 Potomac Ave PA 15216
1,2,57139,Cheapskates,RM 100,Monroeville Mall,Monroeville,PA,15146,Monroeville,116,Chain Packaged Food Only,8,1998-02-06,1998-02-06,NaN,NaN,None,400,0,NaN,-79.796029818,40.4292639820001,RM 100 Monroeville Mall PA 15146
2,3,56986,CVS / Pharmacy #5103,1721,Union Avenue,Natrona Heights,PA,15065,Harrison,116,Chain Packaged Food Only,2,1997-11-03,1997-11-03,2017-03-09,NaN,None,350,7,1,-79.724223,40.622778,1721 Union Avenue PA 15065


## 5. Ingest GeoJSON into MongoDB
**Why MongoDB:** GeoJSON is already a set of JSON documents, with nested `properties` and coordinate arrays of varying length. MongoDB stores each `Feature` unchanged as one document and indexes the geometry natively with a `2dsphere` index, so spatial queries work on the raw data directly.

The file is validated first. A wrong structure, missing geometry or a missing neighborhood name is critical and stops the load. Documents are upserted by feature id, so re-running doesn't create duplicates.

In [9]:
gj = json.loads(GEOJSON_FILE.read_text(encoding="utf-8"))
feature_key = lambda f: f.get("id", (f.get("properties") or {}).get("OBJECTID"))   # file has no top-level id; OBJECTID is the key

# --- validation (critical problems stop here) ---
problems, seen = [], set()
if gj.get("type") != "FeatureCollection" or not gj.get("features"):
    problems.append("not a non-empty FeatureCollection")
for i, f in enumerate(gj.get("features", [])):
    fid = feature_key(f)
    if fid is None or fid in seen: problems.append(f"feature {i}: missing or duplicate id {fid}")
    seen.add(fid)
    if (f.get("geometry") or {}).get("type") not in ("Polygon", "MultiPolygon"): problems.append(f"feature {fid}: bad geometry")
    if not (f.get("properties") or {}).get("hood"): problems.append(f"feature {fid}: no neighborhood name")
if problems:
    raise RuntimeError("CRITICAL GeoJSON problems:\n" + "\n".join(problems))
print(f"{GEOJSON_FILE.name}: {len(gj['features'])} features, validation passed")

# --- load ---
col = mdb["neighborhoods_geojson"]
ids = [feature_key(f) for f in gj["features"]]
res = col.bulk_write([ReplaceOne({"_id": feature_key(f)}, {"_id": feature_key(f), **f}, upsert=True) for f in gj["features"]])
stale = col.delete_many({"_id": {"$nin": ids}}).deleted_count
try:
    col.create_index([("geometry", "2dsphere")], name="geometry_2dsphere")
    index_status = "2dsphere index OK"
except OperationFailure as e:
    index_status = f"WARNING 2dsphere index failed: {e.details.get('errmsg', e)}"
mdb["load_log"].insert_one({"run_id": RUN_ID, "collection": col.name, "source_url": GEOJSON_URL,
                            "file_sha256": sha256(GEOJSON_FILE), "features": len(ids),
                            "loaded_at": datetime.now(timezone.utc)})
print(f"{mdb.name}.{col.name}: {res.upserted_count} inserted, {res.modified_count} updated, "
      f"{stale} stale removed, {col.count_documents({})} total | {index_status}")

neighborhoods.geojson: 90 features, validation passed
group7.neighborhoods_geojson: 0 inserted, 90 updated, 0 stale removed, 90 total | 2dsphere index OK


**Sample document** (coordinates shortened) **and indexes**

In [10]:
doc = col.find_one({}, {"_id": 1, "type": 1, "properties.hood": 1, "properties.hood_no": 1,
                        "properties.acres": 1, "properties.sqmiles": 1, "geometry.type": 1,
                        "geometry.coordinates": {"$slice": 1}})
ring = doc["geometry"]["coordinates"][0]
doc["geometry"]["coordinates"] = [ring[:3] + [f"... {len(ring) - 3} more points"]]
print(json.dumps(doc, indent=2, default=str))
print("indexes:", [ix["key"] for ix in col.list_indexes()])

{
  "_id": 1,
  "type": "Feature",
  "properties": {
    "hood": "Point Breeze North",
    "hood_no": 64,
    "acres": 193.22923916,
    "sqmiles": 0.30192069
  },
  "geometry": {
    "type": "Polygon",
    "coordinates": [
      [
        [
          -79.9053132903415,
          40.4554899761301
        ],
        [
          -79.9048782908077,
          40.4552929758223
        ],
        [
          -79.9037462900863,
          40.4547819762411
        ],
        "... 141 more points"
      ]
    ]
  }
}
indexes: [SON([('_id', 1)]), SON([('geometry', '2dsphere')])]


**A spatial query on the raw documents.** Which neighborhood contains the Cathedral of Learning?

In [11]:
point = {"type": "Point", "coordinates": [-79.9533, 40.4443]}
[d["properties"]["hood"] for d in col.find({"geometry": {"$geoIntersects": {"$geometry": point}}}, {"properties.hood": 1})]

['North Oakland']

## 6. Flatten neighborhoods into PostGIS
MongoDB remains the raw store for the GeoJSON. To join neighborhoods with the inspection data in SQL, the features are:
1. copied from MongoDB unchanged into `raw.neighborhoods_geojson`, one `JSONB` row per feature, then
2. flattened in SQL into `staging.neighborhoods`, with typed columns pulled out of the nested `properties`, a repaired `MultiPolygon` geometry (SRID 4326) and a spatial index.

In [12]:
docs = list(col.find({}))
with pg, pg.cursor() as cur:
    cur.execute("DROP TABLE IF EXISTS raw.neighborhoods_geojson; CREATE TABLE raw.neighborhoods_geojson (feature_id TEXT, doc JSONB)")
    execute_values(cur, "INSERT INTO raw.neighborhoods_geojson VALUES %s",
                   [(str(d.pop("_id")), Json(d)) for d in docs])

sql(r'''
DROP TABLE IF EXISTS staging.neighborhoods;
CREATE TABLE staging.neighborhoods AS
SELECT
    feature_id::int                                          AS neighborhood_id,
    btrim(doc #>> '{properties,hood}')                       AS neighborhood,
    meta.try_int(doc #>> '{properties,hood_no}')::int        AS hood_no,
    meta.try_float(doc #>> '{properties,acres}')::numeric(12,2)   AS acres,
    meta.try_float(doc #>> '{properties,sqmiles}')::numeric(10,4) AS sq_miles,
    NULLIF(doc #>> '{properties,dpwdiv}', '')                AS dpw_division,
    NULLIF(doc #>> '{properties,sectors}', '')               AS sector,
    ST_IsValid(ST_SetSRID(ST_GeomFromGeoJSON(doc ->> 'geometry'), 4326)) AS source_geom_valid,
    ST_Multi(ST_CollectionExtract(ST_MakeValid(
        ST_SetSRID(ST_GeomFromGeoJSON(doc ->> 'geometry'), 4326)), 3))::geometry(MultiPolygon, 4326) AS geom
FROM raw.neighborhoods_geojson;

ALTER TABLE staging.neighborhoods ADD PRIMARY KEY (neighborhood_id);
ALTER TABLE staging.neighborhoods ALTER COLUMN neighborhood SET NOT NULL;
ALTER TABLE staging.neighborhoods ADD CONSTRAINT neighborhoods_name_uq UNIQUE (neighborhood);
CREATE INDEX neighborhoods_geom_gix ON staging.neighborhoods USING gist (geom);
''')
sql('''SELECT neighborhood_id, neighborhood, hood_no, acres, sq_miles, sector, source_geom_valid,
              ST_NPoints(geom) AS polygon_points
       FROM staging.neighborhoods ORDER BY neighborhood LIMIT 8''')

,neighborhood_id,neighborhood,hood_no,acres,sq_miles,sector,source_geom_valid,polygon_points
0,46,Allegheny Center,1,133.72,0.2089,3,True,123
1,33,Allegheny West,2,92.67,0.1448,3,True,110
2,31,Allentown,3,190.08,0.2970,6,True,342
3,50,Arlington,4,307.18,0.4800,7,True,516
4,19,Arlington Heights,5,81.43,0.1272,7,True,327
5,54,Banksville,6,625.68,0.9776,5,True,188
6,4,Bedford Dwellings,7,112.43,0.1757,15,True,138
7,73,Beechview,8,933.89,1.4592,5,True,302


## 7. Data-quality checks
Each check counts the affected rows in the **raw** data and writes a row to `meta.dq_log` with the action taken (requirement 14).

- **Critical** checks protect the keys and joins the pipeline depends on. Any failure **raises an error and stops the notebook**.
- **Warning** checks are recoverable issues that section 8 corrects, excludes or flags.
- **Info** checks are documented observations, kept as-is.

In [13]:
TEST_NAME_REGEX = r"^test\y|\ytest$|\(test|test client|training client|add client"   # test/training accounts, not real facilities
dq_rows = []

def check(table, name, severity, count_sql, action, details=""):
    n = int(sql(count_sql).iloc[0, 0])
    sql('''INSERT INTO meta.dq_log (run_id, table_name, check_name, severity, affected_rows, action_taken, details)
           VALUES (%s, %s, %s, %s, %s, %s, %s)''', (RUN_ID, table, name, severity, n, action, details))
    dq_rows.append({"severity": severity, "table": table, "check": name, "rows": n, "action": action})
    if severity == "critical" and n > 0:
        raise RuntimeError(f"CRITICAL data-quality failure: {name} in {table} ({n:,} rows). Pipeline stopped.")
    return n

def q_test(table):  # SQL to count test-account rows (regex inlined as a literal)
    return f"SELECT count(*) FROM raw.{table} WHERE btrim(facility_name) ~* '{TEST_NAME_REGEX}'"

### 7a. Critical: keys and join integrity

In [14]:
I, V, F = "raw.inspections_2014_2025", "raw.violations_2014_2025", "raw.geocoded_facilities"
STOP = "pipeline stops if > 0"

check(I, "missing_key", "critical", f"SELECT count(*) FROM {I} WHERE meta.try_int(encounter) IS NULL OR meta.try_int(id) IS NULL", STOP, "encounter and facility id must be integers")
check(I, "duplicate_encounter", "critical", f"SELECT count(*) - count(DISTINCT encounter) FROM {I}", STOP, "encounter must uniquely identify an inspection")
check(V, "missing_key", "critical", f"SELECT count(*) FROM {V} WHERE meta.try_int(_id) IS NULL OR meta.try_int(encounter) IS NULL", STOP)
check(V, "orphan_violation", "critical", f"SELECT count(*) FROM {V} v WHERE NOT EXISTS (SELECT 1 FROM {I} i WHERE i.encounter = v.encounter)", STOP, "every violation must belong to a known inspection")
check(F, "duplicate_facility_id", "critical", f"SELECT count(*) - count(DISTINCT id) FROM {F}", STOP)
check(I, "unknown_facility", "critical", f"SELECT count(*) FROM {I} i WHERE NOT EXISTS (SELECT 1 FROM {F} f WHERE f.id = i.id)", STOP, "every inspection must reference a known facility")
check(I, "unparseable_inspection_date", "critical", f"SELECT count(*) FROM {I} WHERE meta.try_date(inspect_dt) IS NULL", STOP)
print("all critical checks passed")

all critical checks passed


### 7b. Recoverable issues and observations

In [15]:
# Test / training accounts
check(I, "test_facility_records", "warning", q_test("inspections_2014_2025"), "excluded from staging", "names like 'Test Client', '(TEST) ...'")
check(V, "test_facility_records", "warning", q_test("violations_2014_2025"), "excluded from staging")
check(F, "test_facility_records", "warning", q_test("geocoded_facilities"), "excluded from staging")

# Inspection times
check(I, "end_before_start_12h_entry", "warning", f'''
    SELECT count(*) FROM {I} WHERE meta.try_time(end_time) < meta.try_time(start_time)
       AND meta.try_time(end_time) + interval '12 hours' >= meta.try_time(start_time)''',
    "end_time corrected by +12h", "e.g. 13:05 -> 01:30 recorded instead of 13:30")
check(I, "end_before_start_other", "warning", f'''
    SELECT count(*) FROM {I} WHERE meta.try_time(end_time) < meta.try_time(start_time)
       AND meta.try_time(end_time) + interval '12 hours' < meta.try_time(start_time)''',
    "end_time set to NULL")
check(I, "missing_end_time", "info", f"SELECT count(*) FROM {I} WHERE NULLIF(btrim(end_time), '') IS NULL", "kept; duration NULL")
check(I, "duration_over_8h", "warning", f'''
    WITH t AS (SELECT meta.try_time(start_time) AS st, meta.try_time(end_time) AS et FROM {I}),
         f AS (SELECT st, CASE WHEN et >= st THEN et WHEN et + interval '12 hours' >= st THEN et + interval '12 hours' END AS ef FROM t)
    SELECT count(*) FROM f WHERE ef - st > interval '8 hours' ''',
    "duration set to NULL (times kept)", "implausibly long inspection")

# Dates
check(F, "business_start_before_1950", "info", f"SELECT count(*) FROM {F} WHERE meta.try_date(bus_st_date) < '1950-01-01'", "kept; likely placeholder dates", "e.g. 1903-07-01")
check(F, "business_start_in_future", "warning", f"SELECT count(*) FROM {F} WHERE meta.try_date(bus_st_date) > current_date", "set to NULL")
check(F, "closed_before_opened", "warning", f"SELECT count(*) FROM {F} WHERE meta.try_date(bus_cl_date) < meta.try_date(bus_st_date)", "close date set to NULL")

# Location
check(F, "missing_coordinates", "warning", f"SELECT count(*) FROM {F} WHERE meta.try_float(x) IS NULL OR meta.try_float(y) IS NULL", "kept; geom NULL, so no neighborhood can be assigned")
check(F, "coordinates_outside_county", "warning", f'''
    SELECT count(*) FROM {F} WHERE meta.try_float(x) IS NOT NULL
       AND NOT (meta.try_float(x) BETWEEN -80.4 AND -79.6 AND meta.try_float(y) BETWEEN 40.1 AND 40.8)''', "geom set to NULL")
check(F, "missing_or_non_pa_state", "info", f"SELECT count(*) FROM {F} WHERE coalesce(upper(btrim(state)), '') <> 'PA'", "kept as-is")
check(F, "invalid_zip", "warning", f"SELECT count(*) FROM {F} WHERE coalesce(btrim(zip), '') !~ '^[0-9]{{5}}'", "set to NULL")
check(F, "same_name_address_multiple_ids", "info", f'''
    SELECT count(*) - count(DISTINCT (lower(btrim(facility_name)), btrim(num), lower(btrim(street)), zip)) FROM {F}''',
    "kept; a new permit (e.g. ownership change) gets a new facility id")
check(F, "untrimmed_text", "info", f"SELECT count(*) FROM {F} WHERE facility_name <> btrim(facility_name) OR street <> btrim(street)", "trimmed in staging")

# Violation severity
check(V, "no_severity_flag", "warning", f"SELECT count(*) FROM {V} WHERE coalesce(low,'') <> 'T' AND coalesce(medium,'') <> 'T' AND coalesce(high,'') <> 'T'", "severity = 'unknown'", "low/medium/high all blank or F")
check(V, "multiple_severity_flags", "info", f"SELECT count(*) FROM {V} WHERE (low='T')::int + (medium='T')::int + (high='T')::int > 1", "severity = highest flag set")
check(V, "rating_not_v", "info", f"SELECT count(*) FROM {V} WHERE rating <> 'V'", "kept; meaning of rating 'N' undocumented")

pd.DataFrame(dq_rows)

,severity,table,check,rows,action
0,critical,raw.inspections_2014_2025,missing_key,0,pipeline stops if > 0
1,critical,raw.inspections_2014_2025,duplicate_encounter,0,pipeline stops if > 0
2,critical,raw.violations_2014_2025,missing_key,0,pipeline stops if > 0
3,critical,raw.violations_2014_2025,orphan_violation,0,pipeline stops if > 0
4,critical,raw.geocoded_facilities,duplicate_facility_id,0,pipeline stops if > 0
5,critical,raw.inspections_2014_2025,unknown_facility,0,pipeline stops if > 0
6,critical,raw.inspections_2014_2025,unparseable_inspection_date,0,pipeline stops if > 0
7,warning,raw.inspections_2014_2025,test_facility_records,190,excluded from staging
8,warning,raw.violations_2014_2025,test_facility_records,958,excluded from staging
9,warning,raw.geocoded_facilities,test_facility_records,50,excluded from staging


## 8. Initial cleaning (`staging` schema)
Each staging table is rebuilt from `raw` with SQL that runs inside the database. The fixes applied here are the actions listed in the quality log above.

| Staging table | Built from | Key | Main fixes |
|---|---|---|---|
| `staging.facilities` | `raw.geocoded_facilities` | `facility_id` | types, trimmed text, 5-digit ZIP, invalid dates nulled, PostGIS point, test accounts removed |
| `staging.inspections` | `raw.inspections_2014_2025` | `inspection_id` (= `encounter`) | types, 12-hour end-time fix, duration, foreign key to facilities |
| `staging.violations` | `raw.violations_2014_2025` | `violation_id` | T/F flags to booleans, derived `severity`, foreign key to inspections |

Facility name and address are kept only in `staging.facilities` and are not repeated on every inspection and violation row.

In [16]:
# Drop in dependency order (violations -> inspections -> facilities) so the rebuild is re-runnable
sql("DROP TABLE IF EXISTS staging.violations, staging.inspections, staging.facilities")

-1

### 8a. `staging.facilities`

In [17]:
sql(r'''
CREATE TABLE staging.facilities AS
SELECT
    meta.try_int(id)                                            AS facility_id,
    btrim(regexp_replace(facility_name, '\s+', ' ', 'g'))       AS facility_name,
    NULLIF(btrim(num), '')                                      AS street_number,
    NULLIF(btrim(regexp_replace(street, '\s+', ' ', 'g')), '')  AS street,
    initcap(NULLIF(btrim(city), ''))                            AS city,
    upper(NULLIF(btrim(state), ''))                             AS state,
    CASE WHEN btrim(zip) ~ '^[0-9]{5}' THEN left(btrim(zip), 5) END AS zip,
    NULLIF(btrim(municipal), '')                                AS municipality,
    meta.try_int(category_cd)::int                              AS category_code,
    NULLIF(btrim(description), '')                              AS category,
    meta.try_int(seat_count)::int                               AS seat_count,
    meta.try_int(sq_feet)::int                                  AS square_feet,
    meta.try_int(status)::int                                   AS status_code,
    CASE WHEN meta.try_date(bus_st_date) <= current_date THEN meta.try_date(bus_st_date) END AS business_start_date,
    CASE WHEN meta.try_date(bus_cl_date) >= meta.try_date(bus_st_date)
           OR meta.try_date(bus_st_date) IS NULL THEN meta.try_date(bus_cl_date) END        AS business_close_date,
    CASE WHEN meta.try_float(x) BETWEEN -80.4 AND -79.6 AND meta.try_float(y) BETWEEN 40.1 AND 40.8
         THEN ST_SetSRID(ST_MakePoint(meta.try_float(x), meta.try_float(y)), 4326)::geometry(Point, 4326)
    END                                                         AS geom
FROM raw.geocoded_facilities
WHERE coalesce(btrim(facility_name), '') !~* %(test)s;

ALTER TABLE staging.facilities ADD PRIMARY KEY (facility_id);
ALTER TABLE staging.facilities ALTER COLUMN facility_name SET NOT NULL;
ALTER TABLE staging.facilities ADD CONSTRAINT facilities_zip_chk CHECK (zip ~ '^[0-9]{5}$');
ALTER TABLE staging.facilities ADD CONSTRAINT facilities_dates_chk CHECK (business_close_date >= business_start_date);
CREATE INDEX facilities_geom_gix ON staging.facilities USING gist (geom);
''', {"test": TEST_NAME_REGEX})
sql("SELECT facility_id, facility_name, street_number, street, city, zip, municipality, category, business_start_date, ST_AsText(geom) AS geom FROM staging.facilities LIMIT 5")

,facility_id,facility_name,street_number,street,city,zip,municipality,category,business_start_date,geom
0,57137,The Pizza Co,1409,Potomac Ave,Pittsburgh,15216,Dormont,Chain Restaurant without Liquor,1998-02-03,POINT(-80.034597 40.395249)
1,57139,Cheapskates,RM 100,Monroeville Mall,Monroeville,15146,Monroeville,Chain Packaged Food Only,1998-02-06,POINT(-79.796029818 40.4292639820001)
2,56986,CVS / Pharmacy #5103,1721,Union Avenue,Natrona Heights,15065,Harrison,Chain Packaged Food Only,1997-11-03,POINT(-79.724223 40.622778)
3,56973,Connie's Place,4613,Liberty Ave,Pittsburgh,15224,Pittsburgh-109,Restaurant without Liquor,1999-07-19,POINT(-79.9503525 40.462569)
4,56974,The Food Depot At Home Depot,3550,Wm Penn Hwy,Pittsburgh,15235,Wilkins,Mobile – Tier I (Unopened Pre-packaged Food Only),1997-11-24,POINT(-79.8053328723998 40.4290899323591)


### 8b. `staging.inspections`

In [18]:
sql(r'''
CREATE TABLE staging.inspections AS
WITH src AS (
    SELECT i.*, meta.try_time(start_time) AS st, meta.try_time(end_time) AS et
    FROM raw.inspections_2014_2025 i
    WHERE meta.try_int(i.id) IN (SELECT facility_id FROM staging.facilities)   -- drops test accounts
), fixed AS (
    SELECT src.*,
           CASE WHEN et >= st THEN et                                               -- normal
                WHEN et + interval '12 hours' >= st THEN et + interval '12 hours'   -- 12-hour entry error
           END AS et_fixed                                                          -- else NULL
    FROM src
)
SELECT
    meta.try_int(encounter)               AS inspection_id,
    meta.try_int(id)                      AS facility_id,
    meta.try_date(inspect_dt)             AS inspection_date,
    st                                    AS start_time,
    et_fixed                              AS end_time,
    CASE WHEN et_fixed - st <= interval '8 hours'
         THEN (extract(epoch FROM et_fixed - st) / 60)::int END AS duration_minutes,
    meta.try_int(placard_st)::int         AS placard_code,
    btrim(placard_desc)                   AS placard,
    meta.try_int(ispt_purpose)::int       AS purpose_code,
    btrim(purpose)                        AS purpose,
    meta.try_int(reispt_cd)::int          AS reinspection_code,
    meta.try_date(reispt_dt)              AS reinspection_due_date,
    meta.try_int(category_cd)::int        AS category_code,
    btrim(description)                    AS category,
    NULLIF(btrim(municipal), '')          AS municipality
FROM fixed;

ALTER TABLE staging.inspections ADD PRIMARY KEY (inspection_id);
ALTER TABLE staging.inspections ALTER COLUMN inspection_date SET NOT NULL;
ALTER TABLE staging.inspections ADD CONSTRAINT inspections_facility_fk FOREIGN KEY (facility_id) REFERENCES staging.facilities;
ALTER TABLE staging.inspections ADD CONSTRAINT inspections_time_chk CHECK (end_time >= start_time);
CREATE INDEX inspections_facility_idx ON staging.inspections (facility_id);
CREATE INDEX inspections_date_idx ON staging.inspections (inspection_date);
''')
sql("SELECT * FROM staging.inspections ORDER BY inspection_date DESC LIMIT 5")

,inspection_id,facility_id,inspection_date,start_time,end_time,duration_minutes,placard_code,placard,purpose_code,purpose,reinspection_code,reinspection_due_date,category_code,category,municipality
0,202507310003,201011190004,2025-07-31,20:20:00,21:25:00,65,1,Inspected & Permitted,52,"Initial, Complaint",1,2025-08-01,212,Chain Restaurant without Liquor,Pittsburgh-102
1,202507300012,4041,2025-07-30,14:00:00,15:05:00,65,1,Inspected & Permitted,11,"Reinspection, Complaint",2,None,411,Hospital/Other Institution,Pittsburgh-104
2,202507300004,30265,2025-07-30,12:00:00,12:30:00,30,1,Inspected & Permitted,2,Reinspection,2,None,111,Supermarket,Sewickley
3,202507300015,38115,2025-07-30,14:35:00,15:55:00,80,1,Inspected & Permitted,1,Initial,2,None,212,Chain Restaurant without Liquor,Moon
4,202507300009,200612060006,2025-07-30,12:40:00,14:25:00,105,1,Inspected & Permitted,8,Complaint,2,None,112,Chain Supermarket,Shaler


### 8c. `staging.violations`

In [19]:
sql(r'''
CREATE TABLE staging.violations AS
SELECT
    meta.try_int(v._id)                    AS violation_id,
    meta.try_int(v.encounter)              AS inspection_id,
    meta.try_int(v.id)                    AS facility_id,
    btrim(v.description_new)               AS violation_category,
    v.rating,
    CASE v.low    WHEN 'T' THEN true WHEN 'F' THEN false END AS low_flag,
    CASE v.medium WHEN 'T' THEN true WHEN 'F' THEN false END AS medium_flag,
    CASE v.high   WHEN 'T' THEN true WHEN 'F' THEN false END AS high_flag,
    CASE WHEN v.high = 'T' THEN 'high' WHEN v.medium = 'T' THEN 'medium'
         WHEN v.low = 'T' THEN 'low' ELSE 'unknown' END       AS severity,
    btrim(v.url)                           AS report_url
FROM raw.violations_2014_2025 v
WHERE meta.try_int(v.encounter) IN (SELECT inspection_id FROM staging.inspections);   -- drops test accounts

ALTER TABLE staging.violations ADD PRIMARY KEY (violation_id);
ALTER TABLE staging.violations ADD CONSTRAINT violations_inspection_fk FOREIGN KEY (inspection_id) REFERENCES staging.inspections;
ALTER TABLE staging.violations ADD CONSTRAINT violations_facility_fk FOREIGN KEY (facility_id) REFERENCES staging.facilities;
ALTER TABLE staging.violations ADD CONSTRAINT violations_one_per_category UNIQUE (inspection_id, violation_category);
ALTER TABLE staging.violations ADD CONSTRAINT violations_severity_chk CHECK (severity IN ('low','medium','high','unknown'));
CREATE INDEX violations_facility_idx ON staging.violations (facility_id);
''')
sql("SELECT violation_id, inspection_id, facility_id, violation_category, rating, low_flag, medium_flag, high_flag, severity FROM staging.violations LIMIT 5")

,violation_id,inspection_id,facility_id,violation_category,rating,low_flag,medium_flag,high_flag,severity
0,287854015,201401020037,201202030011,Certified Food Protection Manager,V,False,False,False,unknown
1,287854016,201401030015,201202090007,Handwashing Facilities,V,False,False,False,unknown
2,287854017,201401030016,200707240007,Hot Holding Temperatures,N,False,False,False,unknown
3,287854018,201401030016,200707240007,Pest Management,V,False,False,False,unknown
4,287854019,201401030016,200707240007,Certified Food Protection Manager,V,False,False,False,unknown


## 9. Results
### 9a. Raw vs. staging row counts

In [20]:
sql('''
SELECT 'facilities' AS entity, (SELECT count(*) FROM raw.geocoded_facilities) AS raw_rows, (SELECT count(*) FROM staging.facilities) AS staging_rows
UNION ALL SELECT 'inspections 2014-2025', (SELECT count(*) FROM raw.inspections_2014_2025), (SELECT count(*) FROM staging.inspections)
UNION ALL SELECT 'violations 2014-2025', (SELECT count(*) FROM raw.violations_2014_2025), (SELECT count(*) FROM staging.violations)
UNION ALL SELECT 'neighborhoods', (SELECT count(*) FROM raw.neighborhoods_geojson), (SELECT count(*) FROM staging.neighborhoods)
''').assign(removed=lambda d: d.raw_rows - d.staging_rows)

,entity,raw_rows,staging_rows,removed
0,facilities,32245,32195,50
1,inspections 2014-2025,124674,124484,190
2,violations 2014-2025,405474,404516,958
3,neighborhoods,90,90,0


### 9b. Keys and constraints now enforced in `staging`

In [21]:
sql('''
SELECT tc.table_name, tc.constraint_type, string_agg(DISTINCT tc.constraint_name, ', ') AS constraints
FROM information_schema.table_constraints tc
WHERE tc.table_schema = 'staging' AND tc.constraint_type IN ('PRIMARY KEY','FOREIGN KEY','UNIQUE','CHECK')
  AND tc.constraint_name NOT LIKE '%_not_null'
GROUP BY 1, 2 ORDER BY 1, 2
''')

,table_name,constraint_type,constraints
0,facilities,CHECK,"facilities_dates_chk, facilities_zip_chk"
1,facilities,PRIMARY KEY,facilities_pkey
2,inspections,CHECK,inspections_time_chk
3,inspections,FOREIGN KEY,inspections_facility_fk
4,inspections,PRIMARY KEY,inspections_pkey
5,neighborhoods,PRIMARY KEY,neighborhoods_pkey
6,neighborhoods,UNIQUE,neighborhoods_name_uq
7,violations,CHECK,violations_severity_chk
8,violations,FOREIGN KEY,"violations_facility_fk, violations_inspection_fk"
9,violations,PRIMARY KEY,violations_pkey


### 9c. Cleaning outcomes

In [22]:
display(sql('''
SELECT severity, count(*) AS violations, round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS pct
FROM staging.violations GROUP BY 1 ORDER BY 2 DESC'''))
display(sql('''
SELECT count(*) AS inspections,
       count(*) FILTER (WHERE end_time IS NULL)         AS end_time_null,
       count(*) FILTER (WHERE duration_minutes IS NULL) AS duration_null,
       round(avg(duration_minutes))                     AS avg_duration_min,
       percentile_cont(0.5) WITHIN GROUP (ORDER BY duration_minutes) AS median_duration_min
FROM staging.inspections'''))

,severity,violations,pct
0,low,270324,66.8
1,medium,70857,17.5
2,high,40666,10.1
3,unknown,22669,5.6


,inspections,end_time_null,duration_null,avg_duration_min,median_duration_min
0,124484,893,2258,59,55.0


### 9d. Data-quality log for this run

In [23]:
sql('''SELECT severity, table_name, check_name, affected_rows, action_taken, details
       FROM meta.dq_log WHERE run_id = %s
       ORDER BY array_position(ARRAY['critical','warning','info'], severity), affected_rows DESC''', (RUN_ID,))

,severity,table_name,check_name,affected_rows,action_taken,details
0,critical,raw.inspections_2014_2025,missing_key,0,pipeline stops if > 0,encounter and facility id must be integers
1,critical,raw.inspections_2014_2025,duplicate_encounter,0,pipeline stops if > 0,encounter must uniquely identify an inspection
2,critical,raw.violations_2014_2025,missing_key,0,pipeline stops if > 0,
3,critical,raw.violations_2014_2025,orphan_violation,0,pipeline stops if > 0,every violation must belong to a known inspection
4,critical,raw.geocoded_facilities,duplicate_facility_id,0,pipeline stops if > 0,
5,critical,raw.inspections_2014_2025,unknown_facility,0,pipeline stops if > 0,every inspection must reference a known facility
6,critical,raw.inspections_2014_2025,unparseable_inspection_date,0,pipeline stops if > 0,
7,warning,raw.violations_2014_2025,no_severity_flag,22749,severity = 'unknown',low/medium/high all blank or F
8,warning,raw.geocoded_facilities,missing_coordinates,2362,"kept; geom NULL, so no neighborhood can be assigned",
9,warning,raw.inspections_2014_2025,duration_over_8h,1371,duration set to NULL (times kept),implausibly long inspection


### 9e. Preview: linking facilities to neighborhoods
This is a first look at the integration planned for Milestone 3: each facility's point is placed in a neighborhood polygon (`ST_Contains`). Facilities outside the City of Pittsburgh match no neighborhood.

In [24]:
display(sql('''
SELECT count(*) AS facilities,
       count(f.geom) AS with_coordinates,
       count(n.neighborhood) AS in_a_pittsburgh_neighborhood
FROM staging.facilities f LEFT JOIN staging.neighborhoods n ON ST_Contains(n.geom, f.geom)'''))
sql('''
SELECT n.neighborhood, count(DISTINCT f.facility_id) AS facilities, count(i.inspection_id) AS inspections_2014_2025
FROM staging.neighborhoods n
JOIN staging.facilities f ON ST_Contains(n.geom, f.geom)
LEFT JOIN staging.inspections i ON i.facility_id = f.facility_id
GROUP BY 1 ORDER BY facilities DESC LIMIT 10''')

,facilities,with_coordinates,in_a_pittsburgh_neighborhood
0,32195,29857,11030


,neighborhood,facilities,inspections_2014_2025
0,Central Business District,1303,5667
1,South Side Flats,676,2938
2,Strip District,573,1986
3,Bloomfield,500,2179
4,North Shore,472,1800
5,North Oakland,402,2100
6,Central Oakland,393,2037
7,Squirrel Hill South,365,2062
8,Shadyside,358,1745
9,East Liberty,277,1103


In [25]:
pg.close()
mongo.close()
print("connections closed")

connections closed
